# DCC Mission 2 — 성능 개선 실험
기존 ResNet18(88.59%)을 보존하고, 작은 Mel 입력의 해상도를 유지하는 모델 두 개를 추가 학습합니다. 마지막 셀에서 기존 모델까지 세 개를 비교해 Validation Accuracy가 가장 높은 체크포인트만 선택합니다.

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'T4 GPU 런타임으로 다시 연결하세요.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/DCC')
CODE_ZIP = DRIVE_ROOT / 'mission02.zip'
FEATURE_DRIVE_ROOT = DRIVE_ROOT / 'mission02_features'
CODE_ROOT = Path('/content/mission02')
FEATURE_ROOT = Path('/content/mission02_features')
BASELINE_ROOT = DRIVE_ROOT / 'mission02_training_output'
SMALLSTEM_ROOT = DRIVE_ROOT / 'mission02_experiment_smallstem'
HYBRID_ROOT = DRIVE_ROOT / 'mission02_experiment_hybrid'
FINAL_ROOT = DRIVE_ROOT / 'mission02_final_model'

assert CODE_ZIP.is_file(), CODE_ZIP
assert (FEATURE_DRIVE_ROOT / 'metadata.json').is_file(), FEATURE_DRIVE_ROOT
assert (BASELINE_ROOT / 'best_model.pt').is_file(), BASELINE_ROOT
!unzip -q -o "{CODE_ZIP}" -d /content
!mkdir -p "{FEATURE_ROOT}"
!cp -R "{FEATURE_DRIVE_ROOT}/." "{FEATURE_ROOT}/"
!pip -q install -r "{CODE_ROOT}/requirements.txt"
!python "{CODE_ROOT}/verify_features.py" "{FEATURE_ROOT}"

## 실험 1 — Small-stem ResNet18
기존 모델의 7x7 stride-2 stem과 max-pooling을 3x3 stride-1 stem으로 바꿔 64x24 Mel 정보를 더 오래 보존합니다.

In [ ]:
!python "{CODE_ROOT}/train.py" --feature-root "{FEATURE_ROOT}" --output-dir "{SMALLSTEM_ROOT}" --model-name resnet18_smallstem --epochs 16 --batch-size 512 --num-workers 2 --learning-rate 2e-4 --dropout 0.25 --label-smoothing 0.03 --mixup-alpha 0.10 --patience 4 --resume

## 실험 2 — ResNet18 + TDNN hybrid
Small-stem 이미지 특징과 시간축 attentive-statistics 특징을 결합해 119대원/신고자의 음향적 차이를 함께 학습합니다.

In [ ]:
!python "{CODE_ROOT}/train.py" --feature-root "{FEATURE_ROOT}" --output-dir "{HYBRID_ROOT}" --model-name hybrid_resnet18_tdnn --epochs 16 --batch-size 512 --num-workers 2 --learning-rate 2e-4 --dropout 0.25 --label-smoothing 0.03 --mixup-alpha 0.10 --patience 4 --resume

## 기존 모델 포함 자동 비교·선택

In [ ]:
!python "{CODE_ROOT}/select_best_model.py" --candidate-dir "{BASELINE_ROOT}" --candidate-dir "{SMALLSTEM_ROOT}" --candidate-dir "{HYBRID_ROOT}" --output-dir "{FINAL_ROOT}"

In [ ]:
import json
comparison = json.loads((FINAL_ROOT / 'experiment_comparison.json').read_text())
print('최종 선택 모델:', comparison['winner']['model_name'])
print('최종 Validation Accuracy:', f"{comparison['winner']['accuracy_tuned'] * 100:.3f}%")
print('최종 체크포인트:', FINAL_ROOT / 'best_model.pt')